Understand the Dataset

Goal: Before training any model, understand the data completely.

In [1]:
import pandas as pd

df = pd.read_csv(
    "../data/Sentences_AllAgree.txt",
    sep="@",
    names=["text", "label"],
    encoding="ISO-8859-1",
    engine="python"
)

In [2]:
from sklearn.model_selection import train_test_split

df["text"] = df["text"].map(lambda x: x.lower() if isinstance(x, str) else x)

train_df, test_df=train_test_split(df, test_size=0.2,random_state=42)
label2id = {"positive": 0, "neutral": 1, "negative": 2}
train_df["label"] = train_df["label"].map(label2id)
test_df["label"] = test_df["label"].map(label2id)


In [3]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

Data Preprocessing
Goal

    Transform the raw dataset into a format that DistilBERT can understand.

In [9]:
from transformers import AutoTokenizer

tokenizer=AutoTokenizer.from_pretrained("distilbert-base-uncased")
def preprocess_function(examples):
    text_encoded = tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128)
    text_encoded["label"] = examples["label"]  # Include label in the processed dataset
    return text_encoded



tokenized_train_dataset = train_dataset.map(preprocess_function, batched=True)
tokenized_test_dataset = test_dataset.map(preprocess_function, batched=True)


Map:   0%|          | 0/453 [00:00<?, ? examples/s]

Goal

Take a pretrained DistilBERT model and teach it financial sentiment classification.

In [10]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=3
)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [12]:
label2id = {
    "negative": 0,
    "neutral": 1,
    "positive": 2
}

id2label = {
    0: "negative",
    1: "neutral",
    2: "positive"
}
model.config.label2id = label2id
model.config.id2label = id2label

Training Configuration

In [13]:
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support
)
import numpy as np

def compute_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="weighted"
    )

    accuracy = accuracy_score(labels, predictions)

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

In [14]:
from transformers import TrainingArguments
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=50,
    load_best_model_at_end=True,
)

In [15]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_test_dataset,
    processing_class=tokenizer,      # If using newer transformers
    compute_metrics=compute_metrics,
)

In [16]:
trainer.train()

TrainOutput(global_step=342, training_loss=0.24752734139648794, metrics={'train_runtime': 1251.0725, 'train_samples_per_second': 4.343, 'train_steps_per_second': 0.273, 'total_flos': 179927052910848.0, 'train_loss': 0.24752734139648794, 'epoch': 3.0})

In [17]:
trainer.evaluate()

{'eval_loss': 0.11904491484165192,
 'eval_accuracy': 0.9602649006622517,
 'eval_precision': 0.9605333712813555,
 'eval_recall': 0.9602649006622517,
 'eval_f1': 0.9602491904487579,
 'eval_runtime': 16.6746,
 'eval_samples_per_second': 27.167,
 'eval_steps_per_second': 1.739,
 'epoch': 3.0}

In [18]:
trainer.save_model("./saved_model")
tokenizer.save_pretrained("./saved_model")

('./saved_model\\tokenizer_config.json',
 './saved_model\\special_tokens_map.json',
 './saved_model\\vocab.txt',
 './saved_model\\added_tokens.json',
 './saved_model\\tokenizer.json')

In [21]:
from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model="./saved_model",
    tokenizer="./saved_model"
)

text = "Tesla  reports record quarterly losss in 2x."

result = classifier(text)

print(result)

[{'label': 'neutral', 'score': 0.8380768895149231}]
